# 116_author_code_artifacts_colab

Author-code mode notebook for strict author-style replication artifacts and figures.

It can:
- build author-like postprocess files,
- build author-like IR files,
- render strict-author paper figures (Figure 2..14) from those artifacts.

This notebook is intended for Colab and local Jupyter runs.


In [ ]:
import os
import sys
import shlex
import pathlib
import subprocess

import torch


def _find_project_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir() and (p / "scripts" / "build_author_postprocess_like.py").is_file() and (p / "scripts" / "build_author_ir_like.py").is_file():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir() and (cand / "scripts" / "build_author_postprocess_like.py").is_file() and (cand / "scripts" / "build_author_ir_like.py").is_file():
        return cand
    raise RuntimeError(
        "Project root not found. On Colab, clone repo first, for example:\n"
        "!cd /content && git clone https://github.com/codist-posist/econml.git"
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# Run configuration
ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float64"
USE_SELECTED = True
CONS_MODE = "author"

# Post-process length (author default: 10000)
POSTPROCESS_T = 10000

# IR settings (author defaults)
IR_NO_STEPS = 1000
IR_NO_STEPS_DECAY = 400
IR_PRESTEPS = 5
IR_GH_N = 3

POLICIES = [
    "taylor",
    "mod_taylor",
    "discretion",
    "commitment",
    "taylor_zlb",
    "mod_taylor_zlb",
    "discretion_zlb",
    "commitment_zlb",
    "taylor_para",
]

print("DEVICE:", DEVICE)
print("DTYPE:", DTYPE)
print("CONS_MODE:", CONS_MODE)
print("POLICIES:", POLICIES)


# Strict-author figure build/display (Figure 2..14)
STRICT_FIG_DIR = str(PROJECT_ROOT / "artifacts" / "paper_figures_nb116_author_strict")
# If True, figure script will regenerate missing postprocess/IR inputs on its own.
STRICT_ENSURE_INPUTS = False

print("STRICT_FIG_DIR:", STRICT_FIG_DIR)
print("STRICT_ENSURE_INPUTS:", STRICT_ENSURE_INPUTS)


In [ ]:
# Build author-like postprocess files policy by policy.
ok_post, skip_post = [], []
for pol in POLICIES:
    cmd = [
        sys.executable,
        "scripts/build_author_postprocess_like.py",
        "--artifacts_root", ARTIFACTS_ROOT,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--T", str(POSTPROCESS_T),
        "--cons-mode", CONS_MODE,
        "--use_selected" if USE_SELECTED else "--no-use_selected",
        "--policies", pol,
    ]
    print("\nRunning:", " ".join(shlex.quote(x) for x in cmd))
    try:
        subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
        ok_post.append(pol)
    except subprocess.CalledProcessError:
        print(f"[skip] postprocess failed for {pol}")
        skip_post.append(pol)

print("\nPostprocess done.")
print("ok:", ok_post)
print("skip:", skip_post)


In [ ]:
# Build author-like IR files policy by policy.
ok_ir, skip_ir = [], []
for pol in POLICIES:
    cmd = [
        sys.executable,
        "scripts/build_author_ir_like.py",
        "--artifacts-root", str(pathlib.Path(ARTIFACTS_ROOT) / "runs"),
        "--policy", pol,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--use-selected", str(USE_SELECTED).lower(),
        "--cons-mode", CONS_MODE,
        "--no-steps", str(IR_NO_STEPS),
        "--no-steps-decay", str(IR_NO_STEPS_DECAY),
        "--presteps", str(IR_PRESTEPS),
        "--gh-n", str(IR_GH_N),
        "--clean-out-dir", "true",
    ]
    print("\nRunning:", " ".join(shlex.quote(x) for x in cmd))
    try:
        subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
        ok_ir.append(pol)
    except subprocess.CalledProcessError:
        print(f"[skip] IR export failed for {pol}")
        skip_ir.append(pol)

print("\nAll IR exports done.")
print("ok:", ok_ir)
print("skip:", skip_ir)


In [ ]:
from src.table2_builder import _load_run_dir

print("\nArtifact check:")
for pol in POLICIES:
    try:
        run_dir = pathlib.Path(_load_run_dir(str(pathlib.Path(ARTIFACTS_ROOT) / "runs"), pol, use_selected=USE_SELECTED, required_files=()))
    except Exception:
        print(f"\n[{pol}] run not found")
        continue
    ppp = run_dir / "author_postprocess"
    irs = run_dir / "IRS"
    print(f"\n[{pol}] {run_dir}")
    print(" postprocess:", (ppp / "simulated_definitions.npz").exists(), (ppp / "simulated_definitions_NT.npz").exists(), (ppp / "simulated_definitions_SS.npz").exists())
    print(" ir:", (irs / "IR_definitions.npz").exists(), (irs / "IR_states.npz").exists())


## Strict-Author Figures
Build and preview Figure 2..14 generated from author-style postprocess/IR artifacts.

## Strict-Author Figures By Cell
Run each figure cell independently. The first run triggers strict-author build; next cells only display selected figures.

In [ ]:
# Helper: build strict-author figures once, then show by figure number.
from IPython.display import display, Image, Markdown

_strict_figures_built = False


def build_strict_author_figures(force_rebuild: bool = False):
    global _strict_figures_built
    if _strict_figures_built and not force_rebuild:
        print("Strict-author figures already built in this session.")
        return

    fig_cmd = [
        sys.executable,
        "scripts/build_paper_figures_author_strict.py",
        "--artifacts-root", ARTIFACTS_ROOT,
        "--fig-dir", STRICT_FIG_DIR,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--use-selected", str(USE_SELECTED).lower(),
        "--ensure-postprocess", str(bool(STRICT_ENSURE_INPUTS)).lower(),
        "--ensure-ir", str(bool(STRICT_ENSURE_INPUTS)).lower(),
        "--cons-mode", CONS_MODE,
    ]

    print("\nRunning:", " ".join(shlex.quote(x) for x in fig_cmd))
    subprocess.run(fig_cmd, cwd=str(PROJECT_ROOT), check=True)
    _strict_figures_built = True
    print("\nStrict-author figures saved to:", STRICT_FIG_DIR)


def show_strict_figure(fig_no: int, width: int = 980):
    fig_path = pathlib.Path(STRICT_FIG_DIR) / f"figure{int(fig_no)}.png"
    if not fig_path.exists():
        raise FileNotFoundError(f"Missing {fig_path}. Run build_strict_author_figures() first.")
    display(Markdown(f"### Figure {int(fig_no)}"))
    display(Image(filename=str(fig_path), width=width))


In [ ]:
# Figure 2
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(2)


In [ ]:
# Figure 3
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(3)


In [ ]:
# Figure 4
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(4)


In [ ]:
# Figure 5
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(5)


In [ ]:
# Figure 6
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(6)


In [ ]:
# Figure 7
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(7)


In [ ]:
# Figure 8
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(8)


In [ ]:
# Figure 9
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(9)


In [ ]:
# Figure 10
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(10)


In [ ]:
# Figure 11
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(11)


In [ ]:
# Figure 12
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(12)


In [ ]:
# Figure 13
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(13)


In [ ]:
# Figure 14
if not _strict_figures_built:
    build_strict_author_figures()
show_strict_figure(14)


In [ ]:
# Figure 1 note
note = pathlib.Path(STRICT_FIG_DIR) / "figure1_note.txt"
if note.exists():
    print(note.read_text(encoding="utf-8"))
else:
    print("figure1_note.txt not found yet. Build figures first.")
